# Semantic Segmentation with DINOv3 EoMT

This notebook demonstrates how to use LightlyTrain for semantic segmentation with our state-of-the-art [EoMT](https://arxiv.org/abs/2503.19108) model built on [DINOv3](https://github.com/facebookresearch/dinov3) backbones, with our publicly released weights trained on the [COCO-Stuff](https://arxiv.org/abs/1612.03716) and [Cityscapes](https://www.cityscapes-dataset.com/) dataset.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lightly-ai/lightly-train/blob/main/examples/notebooks/eomt_semantic_segmentation.ipynb)

> **Important**: When running on Google Colab make sure to select a GPU runtime for faster processing. You can do this by going to `Runtime` > `Change runtime type` and selecting a GPU hardware accelerator.

## Installation

LightlyTrain can be installed directly via `pip`:

In [ ]:
!pip install lightly-train

> **Important**: LightlyTrain is officially supported on
> - Linux: CPU or CUDA
> - MacOS: CPU only
> - Windows (experimental): CPU or CUDA
>
> We are planning to support MPS for MacOS.
>
> Check the [installation instructions](https://docs.lightly.ai/train/stable/installation.html) for more details on installation.

## Train a semantic segmentation model

Training your own semantic segmentation model is straightforward with LightlyTrain.

### Download a dataset

First download a small example dataset. The archive contains the images, single-channel masks and a `config.yaml` with the dataset paths and class names.

In [ ]:
!wget https://github.com/lightly-ai/coco128_yolo/releases/download/v0.0.3/semantic_segmentation.zip && unzip -q semantic_segmentation.zip

The dataset looks like this after the download completes:

```
semantic_segmentation
├── config.yaml
├── images
│   ├── train2017
│   │   ├── 000000000009.jpg
│   │   ├── ...
│   │   └── 000000000536.jpg
│   └── val2017
│       ├── 000000000139.jpg
│       ├── ...
│       └── 000000009891.jpg
└── masks
    ├── train2017
    │   ├── 000000000009.png
    │   ├── ...
    │   └── 000000000536.png
    └── val2017
        ├── 000000000139.png
        ├── ...
        └── 000000009891.png
```

Each mask pixel holds the class id of the image pixel at the same position. Class id `0` means "unlabeled". Paths in `config.yaml` are resolved relative to the config file itself. `data` also accepts a dictionary with the same keys, see the [documentation](https://docs.lightly.ai/train/stable/semantic_segmentation.html) for details.

### Start training

Start training with the `train_semantic_segmentation` function. You only have to specify the output directory, model, and input data; LightlyTrain configures the remaining training parameters and augmentations.

In [ ]:
import lightly_train

lightly_train.train_semantic_segmentation(
    out="out/my_experiment",
    model="dinov3/vits16-eomt-coco",
    steps=100,  # Small number of steps for demonstration, default is 40_000.
    batch_size=4,  # Small batch size for demonstration, default is 16.
    data="semantic_segmentation/config.yaml",
)

Once training completes, the final model checkpoint is saved in `out/my_experiment/exported_models/exported_last.pt`. The best model according to the validation mIoU is saved in `out/my_experiment/exported_models/exported_best.pt`.

## Predict with your own EoMT weights

In [ ]:
!wget -O cat.jpg http://images.cocodataset.org/val2017/000000039769.jpg

### Load the model weights

Load the best model exported during training with LightlyTrain's `load_model` function:

In [ ]:
model = lightly_train.load_model("out/my_experiment/exported_models/exported_best.pt")

### Predict and visualize the semantic mask

Run prediction on the example image again, this time using the weights from your own training run, and visualize the semantic mask:

In [ ]:
import matplotlib.pyplot as plt
import torch
from torchvision.io import read_image
from torchvision.utils import draw_segmentation_masks

masks = model.predict("cat.jpg")

image = read_image("cat.jpg")
masks = torch.stack([masks == class_id for class_id in masks.unique()])
image_with_masks = draw_segmentation_masks(image, masks, alpha=0.6)
plt.imshow(image_with_masks.permute(1, 2, 0))
plt.axis("off")
plt.show()